In [1]:
from moabb.paradigms import  MotorImagery
from moabb.datasets import *

dataset = AlexMI()
sfreq=250
paradigm = MotorImagery(resample=sfreq)
X, y, meta = paradigm.get_data(
     dataset=dataset, 
     subjects=[1],
     return_epochs=False
)
X.shape


Choosing from all possible events
/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 60 events (all good), 0 – 3 s (baseline off), ~11.3 MiB, data loaded,
 'right_hand': 20
 'feet': 20
 'rest': 20>
  warn(f"warnEpochs {epochs}")
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


(60, 16, 750)

In [2]:
import pywt
import numpy as np


# Frequencies of interest
#freqs = np.linspace(8, 32, 8) # 8Hz to 32Hz in steps of 4
freqs = np.arange(8, 32+1)
print(freqs)
wavelet = 'cmor6-1'
center_freq = pywt.central_frequency(wavelet)
scales = center_freq * sfreq / freqs
scales

[ 8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31
 32]


array([31.25      , 27.77777778, 25.        , 22.72727273, 20.83333333,
       19.23076923, 17.85714286, 16.66666667, 15.625     , 14.70588235,
       13.88888889, 13.15789474, 12.5       , 11.9047619 , 11.36363636,
       10.86956522, 10.41666667, 10.        ,  9.61538462,  9.25925926,
        8.92857143,  8.62068966,  8.33333333,  8.06451613,  7.8125    ])

In [3]:
coeffs, freqs_out = pywt.cwt(X, scales, wavelet, sampling_period=1/sfreq)
X_tfr = np.abs(coeffs)**2
X_tfr = np.moveaxis(X_tfr, 0,2)
X_tfr.shape

(60, 16, 25, 750)

In [4]:
import scipy.signal

downsample_factor = 20
n_bins = int(X_tfr.shape[-1]//20)
X_tfr_sub = scipy.signal.resample(X_tfr, n_bins, axis=-1)
X_tfr_sub.shape

(60, 16, 25, 37)

In [5]:
X_tfr_sub

array([[[[ 5.68167129e-01,  1.10021545e+00,  1.12213808e+00, ...,
           1.73702816e+00,  4.52698038e-01, -3.89392793e-02],
         [ 3.48303251e+00,  2.11892866e+00,  1.78678582e+00, ...,
           1.74438649e+01,  1.11077666e+01,  7.10736771e+00],
         [ 9.07312886e+00,  1.02038440e+01,  1.33563937e+01, ...,
           3.45680981e+01,  2.60399522e+01,  1.73431257e+01],
         ...,
         [ 5.42116512e-01,  4.32015473e-01,  5.37633816e-01, ...,
           1.20835980e+00,  1.22849908e+00,  3.00913956e-01],
         [ 4.43459575e-01,  4.78634545e-01,  5.48889353e-01, ...,
           8.88555945e-01,  1.00864762e+00,  5.27596997e-01],
         [ 3.45047544e-01,  4.67706779e-01,  4.29392852e-01, ...,
           5.83918942e-01,  7.59948523e-01,  6.47300468e-01]],

        [[ 8.43667664e-01,  1.52593604e-01,  7.02303864e-01, ...,
           8.44134532e+00,  4.26519843e+00,  2.38755564e+00],
         [ 1.41245405e+00,  2.80624502e-01,  2.19346966e+00, ...,
           1.58249086e